EuroSAT 데이터 다운로드 + 전처리(분할)
torchvision에 내장된 EuroSAT 데이터셋을 계정 인증 없이 바로 받아서 train/val/test로 나눕니다.

# 0. 필요한 패키지 설치

In [ ]:
!pip install split-folders -q

## 1. EuroSAT 다운로드
torchvision이 자동으로 다운로드 + 압축 해제까지 처리

In [ ]:
import os
import torchvision

RAW_DIR = '/content/eurosat_raw'
os.makedirs(RAW_DIR, exist_ok=True)

_ = torchvision.datasets.EuroSAT(root=RAW_DIR, download=True)

print('다운로드 완료')
for cur, dirs, files in os.walk(RAW_DIR):
    depth = cur.replace(RAW_DIR, '').count(os.sep)
    if depth <= 2:
        print(cur)

100%|██████████| 94.3M/94.3M [00:01<00:00, 78.7MB/s]


다운로드 완료
/content/eurosat_raw
/content/eurosat_raw/eurosat
/content/eurosat_raw/eurosat/2750


## 2. 클래스 폴더 자동 탐색
torchvision 버전에 따라 압축 해제 후 폴더 깊이가 달라질 수 있어서,
'하위 폴더들이 전부 이미지를 담고 있는 디렉토리'를 자동으로 찾아 `DATASET_DIR`로 지정합니다.

In [ ]:
IMG_EXT = ('.jpg', '.jpeg', '.png', '.tif', '.tiff')

def find_dataset_dir(root):
    for cur, dirs, files in os.walk(root):
        if not dirs:
            continue
        sub_has_images = []
        for d in dirs:
            sub_path = os.path.join(cur, d)
            has_img = any(f.lower().endswith(IMG_EXT) for f in os.listdir(sub_path))
            sub_has_images.append(has_img)
        if len(dirs) >= 2 and all(sub_has_images):
            return cur
    return None

DATASET_DIR = find_dataset_dir(RAW_DIR)
print('클래스 폴더가 들어있는 경로:', DATASET_DIR)
print('클래스 목록:', sorted(os.listdir(DATASET_DIR)))

클래스 폴더가 들어있는 경로: /content/eurosat_raw/eurosat/2750
클래스 목록: ['AnnualCrop', 'Forest', 'HerbaceousVegetation', 'Highway', 'Industrial', 'Pasture', 'PermanentCrop', 'Residential', 'River', 'SeaLake']


## 3. 클래스별 이미지 수 확인

In [ ]:
def inspect_classes(dataset_dir):
    counts = {}
    for class_name in sorted(os.listdir(dataset_dir)):
        class_path = os.path.join(dataset_dir, class_name)
        if os.path.isdir(class_path):
            n_images = len([f for f in os.listdir(class_path) if f.lower().endswith(IMG_EXT)])
            counts[class_name] = n_images
    return counts

class_counts = inspect_classes(DATASET_DIR)
total = sum(class_counts.values())

print(f'전체 클래스 수: {len(class_counts)}')
print(f'전체 이미지 수: {total}')
print(f'클래스당 평균 이미지 수: {total / len(class_counts):.1f}')
print(f'최소/최대: {min(class_counts.values())} / {max(class_counts.values())}')
print()
for name, cnt in class_counts.items():
    print(f'  {name}: {cnt}')

전체 클래스 수: 10
전체 이미지 수: 27000
클래스당 평균 이미지 수: 2700.0
최소/최대: 2000 / 3000

  AnnualCrop: 3000
  Forest: 3000
  HerbaceousVegetation: 3000
  Highway: 2500
  Industrial: 2500
  Pasture: 2000
  PermanentCrop: 2500
  Residential: 3000
  River: 2500
  SeaLake: 3000


## 4. Train / Val / Test 분할
클래스 비율을 유지하며(stratified) 70/15/15로 나눕니다.
결과 폴더 구조는 Keras의 `image_dataset_from_directory`와 PyTorch의 `ImageFolder` 양쪽에서 그대로 사용 가능합니다.

In [ ]:
import splitfolders

SPLIT_DIR = '/content/eurosat_split'

splitfolders.ratio(
    DATASET_DIR,
    output=SPLIT_DIR,
    seed=42,
    ratio=(0.7, 0.15, 0.15),
    group_prefix=None,
    move=False,
)

print('분할 완료:', SPLIT_DIR)
for split in ['train', 'val', 'test']:
    split_path = os.path.join(SPLIT_DIR, split)
    n = sum(len(files) for _, _, files in os.walk(split_path))
    print(f'  {split}: {n}장')

Copying files: 27000 files [00:04, 5927.51 files/s]

분할 완료: /content/eurosat_split
  train: 18900장
  val: 4050장
  test: 4050장


## 5. Google Drive에 백업
Colab 세션이 끊기면 `/content` 안의 파일이 사라지므로, 분할 결과를 압축해서 Drive에 저장해둡니다.
다음 세션에서는 이 zip만 다시 풀면 되어 처음부터 다시 다운로드/분할할 필요가 없습니다.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
import shutil

DRIVE_TARGET = '/content/drive/MyDrive/eurosat_split.zip'  # 본인 Drive 경로에 맞게 수정

shutil.make_archive(SPLIT_DIR, 'zip', SPLIT_DIR)
shutil.move(f'{SPLIT_DIR}.zip', DRIVE_TARGET)

print('Drive 백업 완료:', DRIVE_TARGET)

Drive 백업 완료: /content/drive/MyDrive/eurosat_split.zip
